In [0]:
# =============================================================================
# Notebook 02 — Composition Parsing
# Goal     : Parse base_compound formulas → element fractions per site (A, B, O/X)
#             and write ml_perovskites.composition
# Input    : ml_perovskites.master
# Output   : Delta table ml_perovskites.composition
# Seed     : N/A
# =============================================================================

# ── 0. Install dependencies (must be the first cell in Databricks) ────────────
# Run this cell alone first, then run the rest of the notebook.
# The cluster restarts the Python interpreter after %pip install.
%pip install pymatgen --quiet

# ── 1. Imports ────────────────────────────────────────────────────────────────
import re
import json
import pandas as pd
import numpy as np
from pymatgen.core import Composition
from pyspark.sql import functions as F

# ── 2. Load master table ──────────────────────────────────────────────────────
master_pd = spark.table("ml_perovskites.master").select(
    "source_row_id", "base_compound", "A_site_paper", "B_site_paper",
    "family_proposed", "domain"
).toPandas()

print(f"Rows loaded: {len(master_pd)}")
print(f"Unique base_compound values: {master_pd['base_compound'].nunique()}")

# ── 3. Formula pre-processor ──────────────────────────────────────────────────
# Handles the four failure patterns found in the data:
#   (1) (ABO3)x(A'B'O3)y  → expand to weighted sum of element amounts
#   (2) El1/3             → replace fraction with decimal (e.g. 1/3 → 0.3333)
#   (3) phase1/phase2     → keep only the first phase (mixed-phase heterostructures)
#   (4) trailing words    → strip non-chemical suffixes (nanosheets, solid solution, etc.)

def _eval_fractions(formula: str) -> str:
    """Replace integer fractions like 1/3, 2/3 with decimals."""
    return re.sub(
        r'(\d+)/(\d+)',
        lambda m: f"{int(m.group(1))/int(m.group(2)):.4f}",
        formula
    )

def _expand_solid_solution(formula: str) -> str:
    """
    Expand (Phase1)x(Phase2)y notation into a single weighted composition.
    E.g. (SrTiO3)0.95(CaCr0.5Nb0.5O3)0.05  →  Sr0.95Ti0.95Ca0.05Cr0.025Nb0.025O3
    Returns None if pattern not matched or expansion fails.
    """
    # Match one or more (formula)fraction blocks
    pattern = r'\(([^()]+)\)([\d.]+)'
    blocks = re.findall(pattern, formula)
    if not blocks:
        return None
    try:
        combined = {}
        for phase_str, frac_str in blocks:
            frac = float(frac_str)
            phase_comp = Composition(_eval_fractions(phase_str))
            for el, amt in phase_comp.as_dict().items():
                combined[el] = combined.get(el, 0.0) + amt * frac
        # Build formula string from combined dict
        parts = "".join(
            f"{el}{round(amt, 4)}" if round(amt, 4) != 1.0 else el
            for el, amt in sorted(combined.items())
        )
        return parts
    except Exception:
        return None

def preprocess_formula(formula: str) -> tuple:
    """
    Returns (cleaned_formula, note) where note describes what was done.
    Handles these patterns (applied in order):
      (a) Strip parenthetical annotations that are NOT stoichiometry
      (b) Strip trailing descriptive words
      (c) Replace "+" separator → keep first phase
      (d) Replace ":" dopant notation → keep base formula
      (e) Mixed-phase "/" between letters → keep first phase
      (f) Variable stoichiometry "(1-x)" / "(x)" → drop variable, keep base
      (g) Solid-solution (Phase)x(Phase)y → expand to weighted sum
      (h) Fraction stoichiometry 1/3 → decimal
    """
    if not isinstance(formula, str) or not formula.strip():
        return "", "empty"

    f = formula.strip()

    # (a) Strip parenthetical annotations that are purely descriptive text
    # e.g. "(partial)", "(N-doped, proton-exchanged from KSr2Nb3O10)"
    # Safe to remove when parentheses contain letters but NO digits-only stoichiometry
    def _strip_text_parens(s):
        # Remove parens whose content is all letters/spaces/punctuation (no digits)
        return re.sub(r'\([^()0-9]*\)', '', s).strip()

    # (b) Strip trailing descriptive words after a space
    f = re.sub(
        r'\s+(nanosheets?|nanoparticles?|solid\s+solution|powder|film|partial)\s*$',
        '', f, flags=re.IGNORECASE
    )
    f = f.strip()

    # (c) "+" separator between phases — keep first phase
    # e.g. "(K)(Nb)O3 + Bi(Ni0.5Ti0.5)O3 solid solution"
    if '+' in f:
        f = f.split('+')[0].strip()
        f = re.sub(r'\s+solid\s+solution\s*$', '', f, flags=re.IGNORECASE).strip()
        # unwrap single-element parens like (K)(Nb)O3 → KNbO3
        f = re.sub(r'\(([A-Za-z]{1,2})\)', r'\1', f)
        return f, "kept_first_phase_plus"

    # (d) ":" dopant notation — keep base formula before colon
    # e.g. "SrTiO3:Rh"
    if re.search(r'[A-Za-z0-9]:(?:[A-Za-z])', f):
        f = f.split(':')[0].strip()
        return f, "stripped_dopant_colon"

    # (e) Mixed-phase "/" between letters (NOT a fraction like 1/3)
    # e.g. "LaTaON2/LaKNaTaO5"
    if re.search(r'[A-Za-z]/[A-Za-z]', f):
        f = f.split('/')[0].strip()
        return f, "kept_first_phase_slash"

    # (f) Variable stoichiometry "(1-x)" or "(x)" or "(2+)" — strip variable terms
    # e.g. "SrTa(1-x)Mg(x)O(2+)N" → "SrTaMgO2N" (approximate, flags as variable)
    if re.search(r'\([^()]*[x\-\+][^()]*\)', f):
        # Remove variable parenthetical blocks entirely
        f = re.sub(r'\([^()]*[x\-\+][^()]*\)', '', f).strip()
        # Also strip any trailing annotation after first space
        f = f.split('(')[0].strip()
        return f, "variable_stoichiometry_stripped"

    # (a) Now safe to strip text-only parens (e.g. "(partial)")
    f_stripped = _strip_text_parens(f)
    if f_stripped != f:
        f = f_stripped
        note_prefix = "text_paren_stripped"
    else:
        note_prefix = "ok"

    # (g) Solid solution: (Phase)x(Phase)y
    if re.search(r'\([^()]+\)[\d.]+', f):
        expanded = _expand_solid_solution(f)
        if expanded:
            return expanded, "solid_solution_expanded"
        first_phase = re.search(r'\(([^()]+)\)', f)
        if first_phase:
            return first_phase.group(1), "solid_solution_first_phase_only"

    # (h) Fraction stoichiometry like 1/3
    if re.search(r'\d/\d', f):
        f = _eval_fractions(f)
        return f, "fractions_expanded"

    return f, note_prefix

# ── 4. Parsing function ───────────────────────────────────────────────────────
def parse_composition(formula: str) -> dict:
    result = {
        "formula_reduced": None,
        "n_elements":      None,
        "parse_ok":        0,
        "parse_error":     None,
        "preprocess_note": None,
    }
    if not isinstance(formula, str) or not formula.strip():
        result["parse_error"] = "empty_or_null"
        return result

    cleaned, note = preprocess_formula(formula)
    result["preprocess_note"] = note

    if not cleaned:
        result["parse_error"] = "empty_after_preprocessing"
        return result

    try:
        comp = Composition(cleaned)
        result["formula_reduced"] = comp.reduced_formula
        result["n_elements"]      = len(comp.elements)
        result["parse_ok"]        = 1
        for el, amt in comp.as_dict().items():
            result[f"el_{el}"] = round(amt, 6)
    except Exception as e:
        result["parse_error"] = str(e)[:120]

    return result

# ── 5. Apply parser ───────────────────────────────────────────────────────────
print("\nParsing compositions...")
parsed_list = [parse_composition(f) for f in master_pd["base_compound"]]
parsed_df = pd.DataFrame(parsed_list)

parsed_df.insert(0, "source_row_id",  master_pd["source_row_id"].values)
parsed_df.insert(1, "base_compound",  master_pd["base_compound"].values)
parsed_df.insert(2, "family_proposed", master_pd["family_proposed"].values)

# ── 6. Parse diagnostics ──────────────────────────────────────────────────────
n_ok   = parsed_df["parse_ok"].sum()
n_fail = len(parsed_df) - n_ok
print(f"\nParse results: {n_ok} OK | {n_fail} failed")

if n_fail > 0:
    print("\nFailed formulas:")
    fail_df = parsed_df[parsed_df["parse_ok"] == 0][
        ["source_row_id", "base_compound", "preprocess_note", "parse_error"]
    ]
    print(fail_df.to_string(index=False))

el_cols = [c for c in parsed_df.columns if c.startswith("el_")]
print(f"\nDistinct elements detected: {len(el_cols)}")
print("Elements:", sorted([c.replace("el_", "") for c in el_cols]))

# ── 7. A/B site assignment ────────────────────────────────────────────────────
# Clean element tokens: strip numeric suffixes, parenthetical notes,
# and descriptive text (e.g. "K (interlayer)" → "K", "Na0.98" → "Na")

def clean_element_token(token: str) -> str:
    """Return clean element symbol, or empty string if not a valid element."""
    # Strip parenthetical annotations
    token = re.sub(r'\s*\(.*?\)', '', token).strip()
    # Strip leading/trailing numeric parts (e.g. "La0.02" → "La")
    token = re.sub(r'^[\d.]+|[\d.]+$', '', token).strip()
    # Keep only letters
    token = re.sub(r'[^A-Za-z]', '', token)
    # Normalize to element symbol format (first letter upper, rest lower)
    if not token:
        return ''
    return token[0].upper() + token[1:].lower()

def parse_site_list(s: str) -> list:
    """Parse '["Sr", "La0.02"]' → ["Sr", "La"] (cleaned symbols)."""
    if not isinstance(s, str) or not s.strip():
        return []
    try:
        raw = json.loads(s)
        items = raw if isinstance(raw, list) else []
    except Exception:
        items = re.findall(r'[A-Za-z][a-z]?[\d.]*', s)
    cleaned = [clean_element_token(i) for i in items]
    return [c for c in cleaned if c]  # drop empty strings

master_pd["A_list"] = master_pd["A_site_paper"].apply(parse_site_list)
master_pd["B_list"] = master_pd["B_site_paper"].apply(parse_site_list)

# Flatten site records
site_records = []
for _, row in master_pd.iterrows():
    for el in row["A_list"]:
        site_records.append({"source_row_id": row["source_row_id"], "site": "A", "element": el})
    for el in row["B_list"]:
        site_records.append({"source_row_id": row["source_row_id"], "site": "B", "element": el})

site_df = pd.DataFrame(site_records)
print(f"\nSite records: {len(site_df)} total")
print("A-site elements:", sorted(site_df[site_df["site"]=="A"]["element"].unique()))
print("B-site elements:", sorted(site_df[site_df["site"]=="B"]["element"].unique()))

# ── 8. Build wide composition table ──────────────────────────────────────────
a_agg = (site_df[site_df["site"]=="A"]
         .groupby("source_row_id")["element"]
         .apply(lambda x: ",".join(sorted(set(x))))
         .rename("A_site_elements"))

b_agg = (site_df[site_df["site"]=="B"]
         .groupby("source_row_id")["element"]
         .apply(lambda x: ",".join(sorted(set(x))))
         .rename("B_site_elements"))

comp_wide = (parsed_df
             .merge(a_agg, on="source_row_id", how="left")
             .merge(b_agg, on="source_row_id", how="left"))

comp_wide["n_A_elements"] = comp_wide["A_site_elements"].apply(
    lambda x: len(x.split(",")) if isinstance(x, str) and x else 0)
comp_wide["n_B_elements"] = comp_wide["B_site_elements"].apply(
    lambda x: len(x.split(",")) if isinstance(x, str) and x else 0)

print(f"\nComposition table shape: {comp_wide.shape}")

# ── 9. Write Delta table ──────────────────────────────────────────────────────
comp_wide_clean = comp_wide.where(pd.notnull(comp_wide), other=None)
comp_spark = spark.createDataFrame(comp_wide_clean)

(comp_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ml_perovskites.composition"))

print("\n✓ Table ml_perovskites.composition written successfully.")
print(f"  Rows   : {spark.table('ml_perovskites.composition').count()}")
print(f"  Columns: {len(spark.table('ml_perovskites.composition').columns)}")

# ── 10. Acceptance criteria ───────────────────────────────────────────────────
print("\n── Acceptance criteria ─────────────────────────────────────────────────")
final = spark.table("ml_perovskites.composition")
n = final.count()
assert n == 691, f"ERROR: expected 691 rows, got {n}"

n_ok_spark   = final.filter(F.col("parse_ok") == 1).count()
parse_rate   = n_ok_spark / n * 100
n_null_a     = final.filter(F.col("A_site_elements").isNull()).count()
n_null_b     = final.filter(F.col("B_site_elements").isNull()).count()

print(f"  Total rows          : {n}")
print(f"  Parsed successfully : {n_ok_spark} ({parse_rate:.1f}%)")
print(f"  Parse failures      : {n - n_ok_spark}")
print(f"  Rows missing A site : {n_null_a}")
print(f"  Rows missing B site : {n_null_b}")
assert parse_rate >= 95.0, f"ERROR: parse rate {parse_rate:.1f}% below 95% threshold"
print("────────────────────────────────────────────────────────────────────────")